# Stage 1 — Monthly ETF Selection with a Strict Five-Year Rule

This notebook is the independent `02_STAGE1` module. It reads the final Pre-Stage 1 monthly liquidity shortlist and local Yahoo CSV files from `01_PRE_STAGE1/Final_Project_v7`. It does **not** download data.

For holding month `m`, it uses observations only through the preceding month-end. An ETF enters Stage 1 only when the holding month is at least **60 calendar months** after its first valid Yahoo adjusted-close month. For example, an ETF first observed in 2013-08 becomes eligible in 2018-08.

Run all cells from top to bottom. Every run creates a new timestamped folder under `02_STAGE1/stage1_results`; existing results are never overwritten. The final file to return is `Stage1_Monthly_ETF_Winners.xlsx`.


## Scoring rules

- CAGR: 20%, higher is better.
- Correlation with SPY: 15%, lower is better.
- Correlation with SOXX: 15%, lower is better.
- Downside beta versus SPY: 15%, lower is better.
- Signed maximum drawdown: 15%, higher (less negative) is better.
- Downside capture ratio: 15%, lower is better.
- 3-month average daily dollar volume: 5%, higher is better.

The best ETF receives one point for the metric; exact ties split the one point equally. Weighted points sum to the total Stage 1 score. An overall score tie is resolved by lower downside capture, lower downside beta, higher CAGR, higher liquidity, then ticker order.


In [ ]:

# Run this cell first. Existing result folders are never overwritten.
from pathlib import Path
from datetime import datetime

def discover_project_root():
    candidates = [
        Path.cwd() / "01_PRE_STAGE1" / "Final_Project_v7",
        Path.cwd().parent / "01_PRE_STAGE1" / "Final_Project_v7",
        Path(r"C:\Users\Kevin\Desktop\量化\finance homework\ETF portfolio\01_PRE_STAGE1\Final_Project_v7"),
    ]
    for candidate in candidates:
        if (candidate / "yahoo_csv").is_dir() and (candidate / "dynamic_monthly_universe_results").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Final_Project_v7. Set PROJECT_ROOT manually to the folder containing yahoo_csv."
    )

PROJECT_ROOT = discover_project_root()  # Pre-Stage 1 data root
STAGE1_ROOT = PROJECT_ROOT.parent.parent / "02_STAGE1"
OUTPUT_PARENT = STAGE1_ROOT / "stage1_results"
OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("run_%Y%m%d_%H%M%S")
OUTPUT_DIR = OUTPUT_PARENT / run_id
suffix = 1
while OUTPUT_DIR.exists():
    OUTPUT_DIR = OUTPUT_PARENT / f"{run_id}_{suffix:02d}"
    suffix += 1

print("Pre-Stage 1 data root:", PROJECT_ROOT)
print("Stage 1 module root:", STAGE1_ROOT)
print("New Stage 1 run folder:", OUTPUT_DIR)
print("No existing result folder will be overwritten.")


In [ ]:
"""Point-in-time Stage 1 ETF selection with a strict 60-calendar-month history rule.

The script reads the final Stage 0 monthly liquidity shortlist and local Yahoo CSVs,
computes the agreed seven metrics using only data available by the prior month-end,
applies winner-takes-weight scoring, and writes auditable CSV/JSON outputs.
"""

from __future__ import annotations

import hashlib
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd


SECTORS = [
    "Pharmaceuticals",
    "Consumer Staples",
    "Energy",
    "Utilities",
    "Capital Markets & Exchanges",
    "Aerospace & Defense",
    "Insurance",
]

CRITERIA = [
    ("Long-term total return", "total_return_cagr", 0.20, "higher"),
    ("Low correlation with SPY", "correlation_spy", 0.15, "lower"),
    ("Low correlation with SOXX", "correlation_soxx", 0.15, "lower"),
    ("Downside Beta", "downside_beta_spy", 0.15, "lower"),
    ("Maximum Drawdown", "maximum_drawdown", 0.15, "higher"),
    ("Downside Capture Ratio", "downside_capture_ratio", 0.15, "lower"),
    ("3M avg dollar volume", "avg_daily_dollar_volume_3m", 0.05, "higher"),
]

HISTORY_MONTHS_REQUIRED = 60
LIQUIDITY_TRADING_DAYS = 63
TIE_RTOL = 1e-12
TIE_ATOL = 1e-12


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()


def month_distance(later: pd.Timestamp, earlier: pd.Timestamp) -> int:
    return (later.year - earlier.year) * 12 + (later.month - earlier.month)


def iso_date(value) -> str | None:
    if value is None or pd.isna(value):
        return None
    return pd.Timestamp(value).strftime("%Y-%m-%d")


def iso_month(value) -> str | None:
    if value is None or pd.isna(value):
        return None
    return pd.Timestamp(value).strftime("%Y-%m")


def load_market_csv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    required = {"Date", "Close", "Adj Close", "Volume"}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    frame["Date"] = pd.to_datetime(frame["Date"], errors="coerce")
    for column in ["Close", "Adj Close", "Volume"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame = frame.dropna(subset=["Date"]).sort_values("Date")
    frame = frame.drop_duplicates(subset=["Date"], keep="last").set_index("Date")
    return frame[["Close", "Adj Close", "Volume"]]


def calculate_metrics(
    ticker: str,
    holding_month: pd.Timestamp,
    formation_cutoff: pd.Timestamp,
    market: dict[str, pd.DataFrame],
) -> dict:
    window_start = holding_month - pd.DateOffset(years=5)
    asset = market[ticker].loc[window_start:formation_cutoff].copy()
    spy = market["SPY"].loc[window_start:formation_cutoff, ["Adj Close"]].rename(
        columns={"Adj Close": "SPY"}
    )
    soxx = market["SOXX"].loc[window_start:formation_cutoff, ["Adj Close"]].rename(
        columns={"Adj Close": "SOXX"}
    )

    asset_prices = asset["Adj Close"].dropna()
    if len(asset_prices) < 2:
        raise ValueError("fewer than two adjusted-close observations in the five-year window")

    elapsed_years = (asset_prices.index[-1] - asset_prices.index[0]).days / 365.2425
    if elapsed_years <= 0:
        raise ValueError("non-positive elapsed period")
    cagr = (asset_prices.iloc[-1] / asset_prices.iloc[0]) ** (1 / elapsed_years) - 1
    drawdown = asset_prices / asset_prices.cummax() - 1
    maximum_drawdown = drawdown.min()

    aligned = pd.concat(
        [asset_prices.rename("asset"), spy["SPY"], soxx["SOXX"]], axis=1, join="inner"
    ).dropna()
    monthly_prices = aligned.resample("ME").last().dropna()
    monthly_returns = monthly_prices.pct_change(fill_method=None).dropna()
    if len(monthly_returns) < 2:
        raise ValueError("fewer than two aligned monthly returns")

    correlation_spy = monthly_returns["asset"].corr(monthly_returns["SPY"])
    correlation_soxx = monthly_returns["asset"].corr(monthly_returns["SOXX"])
    downside = monthly_returns.loc[monthly_returns["SPY"] < 0, ["asset", "SPY"]]
    if len(downside) < 2 or np.isclose(downside["SPY"].var(ddof=1), 0.0):
        raise ValueError("insufficient negative-SPY months for downside beta")
    downside_beta = downside["asset"].cov(downside["SPY"]) / downside["SPY"].var(ddof=1)
    if np.isclose(downside["SPY"].mean(), 0.0):
        raise ValueError("negative-SPY mean return is zero")
    downside_capture = downside["asset"].mean() / downside["SPY"].mean()

    dollar_volume = (
        market[ticker]
        .loc[:formation_cutoff, ["Close", "Volume"]]
        .dropna()
        .assign(dollar_volume=lambda x: x["Close"] * x["Volume"])
        .tail(LIQUIDITY_TRADING_DAYS)
    )
    if dollar_volume.empty:
        raise ValueError("no valid price-volume observations")

    result = {
        "window_start": iso_date(window_start),
        "formation_cutoff": iso_date(formation_cutoff),
        "first_price_used": iso_date(asset_prices.index[0]),
        "last_price_used": iso_date(asset_prices.index[-1]),
        "daily_price_observations": int(len(asset_prices)),
        "monthly_return_observations": int(len(monthly_returns)),
        "negative_spy_months": int(len(downside)),
        "liquidity_observations": int(len(dollar_volume)),
        "total_return_cagr": float(cagr),
        "correlation_spy": float(correlation_spy),
        "correlation_soxx": float(correlation_soxx),
        "downside_beta_spy": float(downside_beta),
        "maximum_drawdown": float(maximum_drawdown),
        "downside_capture_ratio": float(downside_capture),
        "avg_daily_dollar_volume_3m": float(dollar_volume["dollar_volume"].mean()),
        "max_observation_used": iso_date(
            max(asset_prices.index[-1], monthly_prices.index[-1], dollar_volume.index[-1])
        ),
    }
    for key in [criterion[1] for criterion in CRITERIA]:
        if not np.isfinite(result[key]):
            raise ValueError(f"non-finite metric: {key}")
    return result


def write_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def run_stage1(project_root: Path, output_dir: Path) -> Path:
    project_root = Path(project_root).resolve()
    output_dir = Path(output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=False)

    source_run = project_root / "dynamic_monthly_universe_results" / "run_20260805_131031"
    shortlist_path = source_run / "monthly_top_liquidity_shortlists.csv"
    lifecycle_path = source_run / "etf_lifecycle_and_classification_audit.csv"
    yahoo_dir = project_root / "yahoo_csv"

    shortlist = pd.read_csv(shortlist_path)
    shortlist = shortlist.loc[
        shortlist["role"].eq("TOP_LIQUIDITY_SHORTLIST") & shortlist["sleeve"].isin(SECTORS)
    ].copy()
    shortlist["holding_month"] = pd.to_datetime(shortlist["holding_month"] + "-01")
    shortlist["formation_cutoff"] = pd.to_datetime(shortlist["formation_cutoff"])
    shortlist = shortlist.sort_values(["holding_month", "sleeve"]).reset_index(drop=True)

    lifecycle = pd.read_csv(lifecycle_path, low_memory=False)
    name_map = (
        lifecycle.dropna(subset=["symbol"])
        .drop_duplicates("symbol", keep="last")
        .set_index("symbol")["name"]
        .astype(str)
        .to_dict()
    )

    candidate_tickers = sorted(
        {
            ticker.strip()
            for symbols in shortlist["selected_symbols"].dropna().astype(str)
            for ticker in symbols.split(",")
            if ticker.strip()
        }
    )
    requested_tickers = sorted(set(candidate_tickers + ["SPY", "SOXX"]))

    market: dict[str, pd.DataFrame] = {}
    source_quality_rows = []
    for ticker in requested_tickers:
        path = yahoo_dir / f"{ticker}.csv"
        record = {
            "ticker": ticker,
            "file": str(path),
            "exists": path.exists(),
            "sha256": None,
            "rows": 0,
            "first_date": None,
            "last_date": None,
            "status": "MISSING",
            "issue": "",
        }
        if path.exists():
            try:
                frame = load_market_csv(path)
                market[ticker] = frame
                record.update(
                    {
                        "sha256": sha256(path),
                        "rows": int(len(frame)),
                        "first_date": iso_date(frame.index.min()),
                        "last_date": iso_date(frame.index.max()),
                        "status": "OK",
                    }
                )
            except Exception as exc:
                record["status"] = "INVALID"
                record["issue"] = str(exc)
        source_quality_rows.append(record)

    if "SPY" not in market or "SOXX" not in market:
        raise RuntimeError("SPY and SOXX histories are required")

    candidate_rows = []
    metric_rows = []
    metric_lookup: dict[tuple[str, str, str], dict] = {}
    max_observation_dates = []

    for _, row in shortlist.iterrows():
        holding_month = row["holding_month"]
        cutoff = row["formation_cutoff"]
        sector = row["sleeve"]
        symbols = [value.strip() for value in str(row["selected_symbols"]).split(",") if value.strip()]

        for liquidity_rank, ticker in enumerate(symbols, start=1):
            first_trade_date = market[ticker]["Adj Close"].dropna().index.min() if ticker in market else None
            age_months = (
                month_distance(holding_month, first_trade_date)
                if first_trade_date is not None and not pd.isna(first_trade_date)
                else None
            )
            eligible = age_months is not None and age_months >= HISTORY_MONTHS_REQUIRED
            exclusion_reason = ""
            calculation_status = "NOT_RUN"
            if ticker not in market:
                eligible = False
                exclusion_reason = "Missing or invalid Yahoo CSV"
            elif not eligible:
                exclusion_reason = f"History age {age_months} months; requires 60"

            candidate_record = {
                "holding_month": iso_month(holding_month),
                "formation_cutoff": iso_date(cutoff),
                "sector": sector,
                "liquidity_rank": liquidity_rank,
                "ticker": ticker,
                "etf_name": name_map.get(ticker, ""),
                "first_trade_date": iso_date(first_trade_date),
                "first_trade_month": iso_month(first_trade_date),
                "history_age_months": age_months,
                "required_history_months": HISTORY_MONTHS_REQUIRED,
                "eligible_5y": bool(eligible),
                "calculation_status": calculation_status,
                "exclusion_reason": exclusion_reason,
            }

            if eligible:
                try:
                    metrics = calculate_metrics(ticker, holding_month, cutoff, market)
                    calculation_status = "OK"
                    candidate_record["calculation_status"] = calculation_status
                    metric_record = {
                        "holding_month": iso_month(holding_month),
                        "formation_cutoff": iso_date(cutoff),
                        "sector": sector,
                        "liquidity_rank": liquidity_rank,
                        "ticker": ticker,
                        "etf_name": name_map.get(ticker, ""),
                        "first_trade_date": iso_date(first_trade_date),
                        "history_age_months": age_months,
                        **metrics,
                    }
                    metric_rows.append(metric_record)
                    metric_lookup[(iso_month(holding_month), sector, ticker)] = metric_record
                    max_observation_dates.append(pd.Timestamp(metrics["max_observation_used"]))
                except Exception as exc:
                    candidate_record["calculation_status"] = "FAILED"
                    candidate_record["exclusion_reason"] = f"Metric calculation failed: {exc}"
            candidate_rows.append(candidate_record)

    candidates = pd.DataFrame(candidate_rows)
    metrics = pd.DataFrame(metric_rows)

    score_rows = []
    total_rows = []
    winner_rows = []

    for (holding_month, sector), group_candidates in candidates.groupby(
        ["holding_month", "sector"], sort=True
    ):
        scorable = group_candidates.loc[group_candidates["calculation_status"].eq("OK")].copy()
        if scorable.empty:
            winner_rows.append(
                {
                    "holding_month": holding_month,
                    "formation_cutoff": group_candidates["formation_cutoff"].iloc[0],
                    "sector": sector,
                    "winner": "",
                    "winner_total_score": None,
                    "scorable_candidates": 0,
                    "selection_status": "NO_ELIGIBLE_5Y_ETF",
                    "selection_reason": "No Stage 0 candidate had 60 calendar months of history and complete metrics",
                }
            )
            continue

        tickers = scorable["ticker"].tolist()
        total_scores = {ticker: 0.0 for ticker in tickers}
        for criterion_label, metric_key, weight, direction in CRITERIA:
            values = {
                ticker: float(metric_lookup[(holding_month, sector, ticker)][metric_key])
                for ticker in tickers
            }
            best_value = max(values.values()) if direction == "higher" else min(values.values())
            tied = [
                ticker
                for ticker, value in values.items()
                if np.isclose(value, best_value, rtol=TIE_RTOL, atol=TIE_ATOL)
            ]
            tie_count = len(tied)
            metric_winner = ", ".join(sorted(tied))
            for ticker in tickers:
                point = 1.0 / tie_count if ticker in tied else 0.0
                weighted_score = point * weight
                total_scores[ticker] += weighted_score
                score_rows.append(
                    {
                        "holding_month": holding_month,
                        "formation_cutoff": group_candidates["formation_cutoff"].iloc[0],
                        "sector": sector,
                        "ticker": ticker,
                        "criterion": criterion_label,
                        "metric_key": metric_key,
                        "weight": weight,
                        "preferred_direction": direction,
                        "metric_value": values[ticker],
                        "best_value": best_value,
                        "tie_count": tie_count,
                        "point": point,
                        "weighted_score": weighted_score,
                        "metric_winner": metric_winner,
                    }
                )

        ranked = []
        for ticker in tickers:
            metric_record = metric_lookup[(holding_month, sector, ticker)]
            ranked.append(
                {
                    "ticker": ticker,
                    "total_score": total_scores[ticker],
                    "downside_capture_ratio": metric_record["downside_capture_ratio"],
                    "downside_beta_spy": metric_record["downside_beta_spy"],
                    "total_return_cagr": metric_record["total_return_cagr"],
                    "avg_daily_dollar_volume_3m": metric_record["avg_daily_dollar_volume_3m"],
                }
            )
        ranked = sorted(
            ranked,
            key=lambda x: (
                -x["total_score"],
                x["downside_capture_ratio"],
                x["downside_beta_spy"],
                -x["total_return_cagr"],
                -x["avg_daily_dollar_volume_3m"],
                x["ticker"],
            ),
        )
        max_score = ranked[0]["total_score"]
        total_tie_count = sum(
            np.isclose(item["total_score"], max_score, rtol=TIE_RTOL, atol=TIE_ATOL)
            for item in ranked
        )
        winner = ranked[0]["ticker"]
        reason = (
            "Highest weighted total score"
            if total_tie_count == 1
            else "Total-score tie resolved by downside capture, downside beta, CAGR, liquidity, then ticker"
        )
        for rank, item in enumerate(ranked, start=1):
            total_rows.append(
                {
                    "holding_month": holding_month,
                    "formation_cutoff": group_candidates["formation_cutoff"].iloc[0],
                    "sector": sector,
                    "ticker": item["ticker"],
                    "total_score": item["total_score"],
                    "selection_rank": rank,
                    "overall_score_tie_count": total_tie_count,
                    "is_winner": item["ticker"] == winner,
                    "selection_reason": reason if item["ticker"] == winner else "",
                }
            )
        winner_rows.append(
            {
                "holding_month": holding_month,
                "formation_cutoff": group_candidates["formation_cutoff"].iloc[0],
                "sector": sector,
                "winner": winner,
                "winner_total_score": max_score,
                "scorable_candidates": len(tickers),
                "selection_status": "READY",
                "selection_reason": reason,
            }
        )

    scores = pd.DataFrame(score_rows)
    totals = pd.DataFrame(total_rows)
    winners = pd.DataFrame(winner_rows)

    holding_months = sorted(shortlist["holding_month"].dt.strftime("%Y-%m").unique().tolist())
    winner_matrix = []
    for holding_month in holding_months:
        row = {"holding_month": holding_month}
        month_winners = winners.loc[winners["holding_month"].eq(holding_month)]
        for sector in SECTORS:
            match = month_winners.loc[month_winners["sector"].eq(sector), "winner"]
            row[sector] = match.iloc[0] if len(match) else ""
        winner_matrix.append(row)

    # Audits
    future_violations = 0
    if not metrics.empty:
        future_violations = int(
            (
                pd.to_datetime(metrics["max_observation_used"])
                > pd.to_datetime(metrics["formation_cutoff"])
            ).sum()
        )
    ineligible_scored = int(
        candidates.loc[~candidates["eligible_5y"], "calculation_status"].eq("OK").sum()
    )
    score_group_sums = (
        scores.groupby(["holding_month", "sector", "criterion"], as_index=False)[
            "weighted_score"
        ].sum()
        if not scores.empty
        else pd.DataFrame()
    )
    weight_map = {label: weight for label, _, weight, _ in CRITERIA}
    score_weight_failures = 0
    if not score_group_sums.empty:
        score_weight_failures = int(
            sum(
                not np.isclose(row["weighted_score"], weight_map[row["criterion"]])
                for _, row in score_group_sums.iterrows()
            )
        )
    total_group_sums = (
        totals.groupby(["holding_month", "sector"])["total_score"].sum()
        if not totals.empty
        else pd.Series(dtype=float)
    )
    total_score_sum_failures = int(sum(not np.isclose(value, 1.0) for value in total_group_sums))
    ready_winners = int(winners["selection_status"].eq("READY").sum())
    expected_winners = len(holding_months) * len(SECTORS)
    total_score_tie_groups = int(
        winners.loc[winners["selection_status"].eq("READY"), "selection_reason"]
        .str.startswith("Total-score tie")
        .sum()
    )

    audit = {
        "holding_month_start": holding_months[0],
        "holding_month_end": holding_months[-1],
        "holding_month_count": len(holding_months),
        "sector_count": len(SECTORS),
        "expected_winner_cells": expected_winners,
        "ready_winner_cells": ready_winners,
        "blank_winner_cells": expected_winners - ready_winners,
        "candidate_rows": int(len(candidates)),
        "eligible_candidate_rows": int(candidates["eligible_5y"].sum()),
        "ineligible_candidate_rows": int((~candidates["eligible_5y"]).sum()),
        "metric_rows": int(len(metrics)),
        "score_rows": int(len(scores)),
        "total_rows": int(len(totals)),
        "future_data_violations": future_violations,
        "ineligible_candidates_scored": ineligible_scored,
        "metric_weight_sum_failures": score_weight_failures,
        "group_total_score_sum_failures": total_score_sum_failures,
        "total_score_tie_groups": total_score_tie_groups,
        "criteria_weight_sum": float(sum(item[2] for item in CRITERIA)),
        "source_csv_count": int(len(source_quality_rows)),
        "source_csv_ok": int(sum(row["status"] == "OK" for row in source_quality_rows)),
        "source_csv_not_ok": int(sum(row["status"] != "OK" for row in source_quality_rows)),
    }

    candidates.to_csv(output_dir / "stage1_candidate_eligibility.csv", index=False)
    metrics.to_csv(output_dir / "stage1_metric_calculations.csv", index=False)
    scores.to_csv(output_dir / "stage1_weighted_scorecard.csv", index=False)
    totals.to_csv(output_dir / "stage1_monthly_totals.csv", index=False)
    winners.to_csv(output_dir / "stage1_winner_detail.csv", index=False)
    pd.DataFrame(winner_matrix).to_csv(output_dir / "stage1_winner_matrix.csv", index=False)
    pd.DataFrame(source_quality_rows).to_csv(output_dir / "stage1_source_quality.csv", index=False)

    payload = {
        "metadata": {
            "generated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
            "project_root": str(project_root),
            "shortlist_source": str(shortlist_path),
            "shortlist_source_sha256": sha256(shortlist_path),
            "history_rule": "Eligible when holding month is at least 60 calendar months after the first valid Yahoo adjusted-close month",
            "window_rule": "Five-year calendar window ending at the immediately preceding month-end",
            "history_months_required": HISTORY_MONTHS_REQUIRED,
            "liquidity_trading_days": LIQUIDITY_TRADING_DAYS,
            "tie_tolerance_rtol": TIE_RTOL,
            "tie_tolerance_atol": TIE_ATOL,
            "criteria": [
                {
                    "criterion": label,
                    "metric_key": metric,
                    "weight": weight,
                    "direction": direction,
                }
                for label, metric, weight, direction in CRITERIA
            ],
            "overall_tie_break": [
                "lower downside capture ratio",
                "lower downside beta",
                "higher CAGR",
                "higher 3M average daily dollar volume",
                "alphabetically earlier ticker",
            ],
        },
        "audit": audit,
        "sector_order": SECTORS,
        "winner_matrix": winner_matrix,
        "winner_detail": winners.to_dict(orient="records"),
        "candidate_eligibility": candidates.to_dict(orient="records"),
        "metrics": metrics.to_dict(orient="records"),
        "scores": scores.to_dict(orient="records"),
        "totals": totals.to_dict(orient="records"),
        "source_quality": source_quality_rows,
    }
    write_json(output_dir / "stage1_payload.json", payload)
    write_json(
        output_dir / "stage1_run_manifest.json",
        {
            "generated_at": payload["metadata"]["generated_at"],
            "methodology": payload["metadata"],
            "audit": audit,
            "output_files": sorted(path.name for path in output_dir.iterdir()),
        },
    )

    print(json.dumps(audit, indent=2))
    print(f"OUTPUT_DIR={output_dir}")


    return output_dir


In [ ]:

# This is the calculation step. It may take a few minutes.
run_stage1(PROJECT_ROOT, OUTPUT_DIR)

winner_matrix_preview = pd.read_csv(OUTPUT_DIR / "stage1_winner_matrix.csv")
audit_preview = json.loads((OUTPUT_DIR / "stage1_payload.json").read_text(encoding="utf-8"))["audit"]

display(winner_matrix_preview.head(12))
display(pd.DataFrame([audit_preview]).T.rename(columns={0: "Value"}))


In [ ]:

# Build the final XLSX and preserve every intermediate calculation as a separate sheet/file.
from copy import copy
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

winner_matrix = pd.read_csv(OUTPUT_DIR / "stage1_winner_matrix.csv")
winner_detail = pd.read_csv(OUTPUT_DIR / "stage1_winner_detail.csv")
eligibility = pd.read_csv(OUTPUT_DIR / "stage1_candidate_eligibility.csv")
metrics = pd.read_csv(OUTPUT_DIR / "stage1_metric_calculations.csv")
scores = pd.read_csv(OUTPUT_DIR / "stage1_weighted_scorecard.csv")
totals = pd.read_csv(OUTPUT_DIR / "stage1_monthly_totals.csv")
source_quality = pd.read_csv(OUTPUT_DIR / "stage1_source_quality.csv")
payload = json.loads((OUTPUT_DIR / "stage1_payload.json").read_text(encoding="utf-8"))

criteria_table = pd.DataFrame(payload["metadata"]["criteria"])
audit_table = pd.DataFrame(
    [{"Check": key, "Value": value} for key, value in payload["audit"].items()]
)
formula_table = pd.DataFrame(
    [
        ["Total-return CAGR", "(Ending adjusted close / Beginning adjusted close)^(1 / elapsed years) - 1"],
        ["Correlation with SPY", "Correlation of ETF and SPY month-end total returns"],
        ["Correlation with SOXX", "Correlation of ETF and SOXX month-end total returns"],
        ["Downside beta", "Cov(ETF, SPY | SPY monthly return < 0) / Var(SPY | SPY monthly return < 0)"],
        ["Maximum drawdown", "Minimum of adjusted close / running peak - 1; higher (less negative) is better"],
        ["Downside capture", "Mean ETF return in negative-SPY months / Mean SPY return in those months"],
        ["3M avg dollar volume", "Mean of Close × Volume over the final 63 valid trading days"],
        ["Five-year eligibility", "Holding month minus first valid Yahoo adjusted-close month must be at least 60 calendar months"],
        ["Point-in-time rule", "Holding month m uses observations no later than the final trading day of month m−1"],
    ],
    columns=["Item", "Definition"],
)

xlsx_path = OUTPUT_DIR / "Stage1_Monthly_ETF_Winners.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    winner_matrix.to_excel(writer, sheet_name="Stage1 Winners", startrow=3, index=False)
    winner_detail.to_excel(writer, sheet_name="Winner Detail", startrow=3, index=False)
    eligibility.to_excel(writer, sheet_name="Candidate Eligibility", startrow=3, index=False)
    metrics.to_excel(writer, sheet_name="Metric Calculations", startrow=3, index=False)
    scores.to_excel(writer, sheet_name="Weighted Scorecard", startrow=3, index=False)
    totals.to_excel(writer, sheet_name="Monthly Totals", startrow=3, index=False)
    source_quality.to_excel(writer, sheet_name="Source Quality", startrow=3, index=False)
    criteria_table.to_excel(writer, sheet_name="Methodology & Audit", startrow=4, index=False)
    formula_table.to_excel(writer, sheet_name="Methodology & Audit", startrow=15, index=False)
    audit_table.to_excel(writer, sheet_name="Methodology & Audit", startrow=27, index=False)

wb = load_workbook(xlsx_path)
navy = "17365D"
blue = "1F4E78"
teal = "0F6B78"
white = "FFFFFF"
light_blue = "DDEBF7"
band_blue = "D9EAF7"
light_gray = "F3F4F6"
green = "E2F0D9"
amber = "FFF2CC"
border_color = "D7DEE8"
thin = Side(style="thin", color=border_color)

titles = {
    "Stage1 Winners": (
        "Stage 1 Monthly ETF Winners by Sector",
        "Each holding month uses only the preceding month-end information. ETFs participate only after 60 calendar months from their first valid Yahoo adjusted-close month.",
    ),
    "Winner Detail": ("Stage 1 Winner Detail", "One winner record per holding month and sector, including tie-break disclosure."),
    "Candidate Eligibility": ("Candidate Five-Year Eligibility", "Every Stage 0 shortlist candidate is retained, including ETFs excluded for insufficient history."),
    "Metric Calculations": ("Five-Year Metric Calculations", "Python-calculated metrics for eligible candidates only; all dates and observation counts are retained."),
    "Weighted Scorecard": ("Weighted Metric Scorecard", "Winner-takes-weight scoring: metric winner receives one point, divided equally for exact ties, then multiplied by the stated weight."),
    "Monthly Totals": ("Monthly Weighted Totals", "Candidate total scores, final ranks, and overall tie-break outcomes."),
    "Source Quality": ("Source Data Quality and Hashes", "Local Yahoo CSV coverage and SHA-256 hashes used for this run."),
    "Methodology & Audit": ("Stage 1 Methodology & Audit Trail", "Rules, formulas, validation checks, and source references for this exact run."),
}

for ws in wb.worksheets:
    ws.sheet_view.showGridLines = False
    max_col = max(ws.max_column, 2)
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=max_col)
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=max_col)
    title, subtitle = titles[ws.title]
    ws.cell(1, 1, title)
    ws.cell(2, 1, subtitle)
    ws.cell(1, 1).fill = PatternFill("solid", fgColor=navy)
    ws.cell(1, 1).font = Font(color=white, bold=True, size=16)
    ws.cell(1, 1).alignment = Alignment(vertical="center")
    ws.row_dimensions[1].height = 30
    ws.cell(2, 1).fill = PatternFill("solid", fgColor=light_blue)
    ws.cell(2, 1).font = Font(color="1F2937", italic=True, size=10)
    ws.cell(2, 1).alignment = Alignment(wrap_text=True, vertical="center")
    ws.row_dimensions[2].height = 32
    ws.freeze_panes = "B5"

    if ws.title != "Methodology & Audit":
        header_row = 4
        for cell in ws[header_row]:
            cell.fill = PatternFill("solid", fgColor=blue)
            cell.font = Font(color=white, bold=True, size=9)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = Border(bottom=Side(style="medium", color=navy))
        ws.row_dimensions[header_row].height = 34
        ws.auto_filter.ref = f"A4:{get_column_letter(ws.max_column)}{ws.max_row}"
        for row_number in range(5, ws.max_row + 1):
            if row_number % 2 == 1:
                for cell in ws[row_number]:
                    cell.fill = PatternFill("solid", fgColor=band_blue)
            for cell in ws[row_number]:
                cell.border = Border(bottom=thin)
                cell.alignment = Alignment(vertical="center", wrap_text=False)

# Match the preceding matrix workbook closely.
ws = wb["Stage1 Winners"]
ws.freeze_panes = "B5"
for row_number in range(5, ws.max_row + 1):
    ws.row_dimensions[row_number].height = 25
    ws.cell(row_number, 1).font = Font(bold=True, color="1F2937")
    ws.cell(row_number, 1).alignment = Alignment(horizontal="center", vertical="center")
    for col_number in range(2, 9):
        ws.cell(row_number, col_number).alignment = Alignment(horizontal="center", vertical="center")
        if ws.cell(row_number, col_number).value in (None, ""):
            ws.cell(row_number, col_number).fill = PatternFill("solid", fgColor=amber)
ws.column_dimensions["A"].width = 14
for col in "BCDEH":
    ws.column_dimensions[col].width = 20
ws.column_dimensions["F"].width = 24
ws.column_dimensions["G"].width = 23

# Semantic number formats.
def set_column_format(ws, header_name, number_format):
    headers = {cell.value: cell.column for cell in ws[4]}
    if header_name not in headers:
        return
    col = headers[header_name]
    for row in range(5, ws.max_row + 1):
        ws.cell(row, col).number_format = number_format

for sheet_name in ["Metric Calculations"]:
    sheet = wb[sheet_name]
    for header in ["total_return_cagr", "maximum_drawdown"]:
        set_column_format(sheet, header, "0.00%;[Red](0.00%);-")
    for header in ["correlation_spy", "correlation_soxx", "downside_beta_spy", "downside_capture_ratio"]:
        set_column_format(sheet, header, "0.000")
    set_column_format(sheet, "avg_daily_dollar_volume_3m", '"$"#,##0;[Red]("$"#,##0);-')

sheet = wb["Weighted Scorecard"]
for header in ["weight", "point", "weighted_score"]:
    set_column_format(sheet, header, "0.00%;[Red](0.00%);-")
sheet = wb["Monthly Totals"]
set_column_format(sheet, "total_score", "0.00%;[Red](0.00%);-")
sheet = wb["Winner Detail"]
set_column_format(sheet, "winner_total_score", "0.00%;[Red](0.00%);-")

# Practical column widths for audit sheets.
widths = {
    "Winner Detail": [13, 14, 26, 11, 16, 18, 21, 62],
    "Candidate Eligibility": [13, 14, 26, 12, 10, 36, 14, 14, 14, 14, 12, 18, 52],
    "Metric Calculations": [13, 14, 26, 12, 10, 34, 14, 13, 12, 14, 14, 14, 14, 14, 14, 16, 16, 15, 15, 15, 15, 18, 14],
    "Weighted Scorecard": [13, 14, 26, 10, 26, 28, 10, 12, 16, 16, 10, 10, 14, 18],
    "Monthly Totals": [13, 14, 26, 10, 13, 12, 16, 12, 68],
    "Source Quality": [10, 70, 10, 66, 12, 14, 14, 12, 45],
}
for sheet_name, sheet_widths in widths.items():
    sheet = wb[sheet_name]
    for index, width in enumerate(sheet_widths, start=1):
        sheet.column_dimensions[get_column_letter(index)].width = width
    for row in range(5, sheet.max_row + 1):
        sheet.row_dimensions[row].height = 20

# Build readable methodology sections.
ws = wb["Methodology & Audit"]
ws.freeze_panes = "A5"
for section_row, label in [(4, "Scoring Criteria"), (15, "Formula Definitions"), (27, "Run Audit Checks")]:
    ws.merge_cells(start_row=section_row, start_column=1, end_row=section_row, end_column=4)
    ws.cell(section_row, 1, label)
    ws.cell(section_row, 1).fill = PatternFill("solid", fgColor=teal)
    ws.cell(section_row, 1).font = Font(color=white, bold=True)
for header_row in [5, 16, 28]:
    for cell in ws[header_row]:
        if cell.value is not None:
            cell.fill = PatternFill("solid", fgColor=blue)
            cell.font = Font(color=white, bold=True)
            cell.alignment = Alignment(wrap_text=True)
for row in range(6, ws.max_row + 1):
    for cell in ws[row]:
        cell.alignment = Alignment(vertical="top", wrap_text=True)
        if cell.value is not None:
            cell.border = Border(bottom=thin)
ws.column_dimensions["A"].width = 34
ws.column_dimensions["B"].width = 78
ws.column_dimensions["C"].width = 18
ws.column_dimensions["D"].width = 18

# Add source URLs and input hashes below the audit table.
source_start = ws.max_row + 3
ws.merge_cells(start_row=source_start, start_column=1, end_row=source_start, end_column=4)
ws.cell(source_start, 1, "Sources and Reproducibility")
ws.cell(source_start, 1).fill = PatternFill("solid", fgColor=teal)
ws.cell(source_start, 1).font = Font(color=white, bold=True)
source_rows = [
    ["Stage 0 shortlist CSV", payload["metadata"]["shortlist_source"], payload["metadata"]["shortlist_source_sha256"], "Local final v7 output"],
    ["Yahoo Finance chart data", "Daily adjusted close, close and volume", "", "https://query1.finance.yahoo.com/v8/finance/chart/{TICKER}"],
    ["Alpha Vantage LISTING_STATUS", "ETF lifecycle and classification", "", "https://www.alphavantage.co/documentation/#listing-status"],
]
for r, values in enumerate(source_rows, start=source_start + 1):
    for c, value in enumerate(values, start=1):
        ws.cell(r, c, value)
        ws.cell(r, c).alignment = Alignment(wrap_text=True, vertical="top")
        ws.cell(r, c).border = Border(bottom=thin)

for ws in wb.worksheets:
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 0

wb.save(xlsx_path)
print("Created workbook:", xlsx_path)


In [ ]:

# Archive the exact notebook and refresh the manifest, then create a ZIP for convenient return.
import hashlib
import shutil

NOTEBOOK_NAME = "02_STAGE1_rolling_five_year_etf_selection.ipynb"
notebook_source = STAGE1_ROOT / NOTEBOOK_NAME
if notebook_source.exists():
    shutil.copy2(notebook_source, OUTPUT_DIR / NOTEBOOK_NAME)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest().upper()

manifest_path = OUTPUT_DIR / "stage1_run_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["output_files"] = [
    {
        "name": path.name,
        "bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    }
    for path in sorted(OUTPUT_DIR.iterdir())
    if path.is_file() and path.name != manifest_path.name
]
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_base = OUTPUT_PARENT / OUTPUT_DIR.name
zip_path = Path(shutil.make_archive(str(zip_base), "zip", OUTPUT_DIR))

print("\nStage 1 completed.")
print("Workbook to return:", OUTPUT_DIR / "Stage1_Monthly_ETF_Winners.xlsx")
print("Full audit archive:", zip_path)
print("Run folder:", OUTPUT_DIR)
